# Laboratório 10: O Pipeline Definitivo (RAG, QLoRA e Otimização de Inferência na GPU)

> **Partes deste laboratório foram geradas/complementadas com IA, revisadas e validadas por Guilherme Pereira**

Pipeline ponta a ponta: RAG massivo → Modelo quantizado em 4-bits → Geração otimizada com KV Cache + FlashAttention-2.

## Setup: Instalar dependências

In [ ]:
# Instalar dependências necessárias
!pip install -q transformers bitsandbytes accelerate torch
!pip install -q flash-attn --no-build-isolation

In [ ]:
import torch
import time
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

def bytes_to_mb(b):
    return b / (1024 ** 2)

def reset_cuda_stats():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        gc.collect()

print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {bytes_to_mb(torch.cuda.get_device_properties(0).total_memory):.1f} MB")

## Passo 1: Ingestão Eficiente — Carregamento QLoRA em 4-bits

In [ ]:
reset_cuda_stats()
vram_antes_modelo = bytes_to_mb(torch.cuda.memory_allocated()) if torch.cuda.is_available() else 0

# Configuração QLoRA: 4-bit quantization com compute em float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,   # double quantization reduz ainda mais o uso de memória
    bnb_4bit_quant_type="nf4"          # NormalFloat4: melhor para pesos LLM
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

vram_modelo_mb = bytes_to_mb(torch.cuda.memory_allocated()) if torch.cuda.is_available() else 0
vram_modelo_usada = vram_modelo_mb - vram_antes_modelo

print(f"\n=== MÉTRICA PASSO 1 ===")
print(f"VRAM ocupada pelo modelo quantizado (4-bit): {vram_modelo_usada:.1f} MB")
print(f"(Equivalente em float16 seria ~2.200 MB — redução de ~75%)")

## Passo 2: Simulando o RAG Massivo (~10.000–15.000 tokens)

In [ ]:
# Texto fictício simulando 5 capítulos de manuais médicos recuperados pelo RAG
CAPITULO_TEMPLATE = """
Capítulo {n}: {titulo}

1. Fisiopatologia
A fisiopatologia desta condição envolve uma cascata inflamatória mediada por citocinas pró-inflamatórias,
incluindo TNF-α, IL-6 e IL-1β. A ativação do complemento desempenha papel central na progressão da doença,
com consequente disfunção endotelial e aumento da permeabilidade vascular. Estudos de coorte prospectivos
demonstraram correlação positiva entre marcadores séricos de inflamação (PCR, VHS, ferritina) e desfechos
clínicos adversos. A disfunção mitocondrial observada em biópsias teciduais sugere comprometimento da
fosforilação oxidativa, resultando em estresse oxidativo celular com produção exacerbada de espécies reativas
de oxigênio (ROS). O comprometimento da barreira hematoencefálica, quando presente, correlaciona-se com
manifestações neurológicas incluindo encefalopatia metabólica, convulsões e alterações cognitivas.

2. Diagnóstico Diferencial
O diagnóstico diferencial deve considerar etiologias infecciosas, autoimunes, neoplásicas e metabólicas.
A investigação laboratorial de primeira linha inclui hemograma completo com diferencial, painel metabólico
completo, função hepática e renal, coagulograma, marcadores inflamatórios e sorologias direcionadas.
Exames de imagem como tomografia computadorizada com contraste e ressonância magnética são indicados
quando há suspeita de acometimento de órgãos específicos. A biopsia tecidual com análise histopatológica
e imunohistoquímica permanece o padrão ouro para confirmação diagnóstica em casos selecionados.
A análise citogenética e molecular é essencial no contexto de doenças hematológicas e neoplasias sólidas.

3. Protocolo Terapêutico
O manejo terapêutico baseia-se em evidências de ensaios clínicos randomizados (ECRs) de alta qualidade.
A terapia de primeira linha consiste em corticosteroides sistêmicos (prednisona 1 mg/kg/dia) com redução
gradual conforme resposta clínica e laboratorial. Imunossupressores como azatioprina, micofenolato mofetil
e ciclofosfamida são reservados para casos refratários ou com contraindicação aos corticosteroides.
Agentes biológicos direcionados, incluindo inibidores de TNF-α (infliximabe, adalimumabe, etanercepte),
inibidores de IL-6 (tocilizumabe, sarilumabe) e inibidores de JAK (baricitinibe, tofacitinibe), representam
avanços terapêuticos significativos para subgrupos específicos de pacientes.

4. Monitoramento e Desfechos
O monitoramento da resposta terapêutica deve ser realizado com avaliações clínicas seriadas e reavaliação
laboratorial a cada 4-8 semanas. Critérios de remissão incluem ausência de atividade clínica, normalização
de marcadores inflamatórios e estabilização de parâmetros funcionais. Desfechos adversos incluem falência
orgânica, infecções oportunistas secundárias à imunossupressão e toxicidade medicamentosa. A qualidade
de vida relacionada à saúde (QVRS) deve ser avaliada por instrumentos validados como SF-36, EQ-5D e
escalas específicas da doença. O seguimento de longo prazo é essencial para detecção precoce de recidivas.

5. Considerações Especiais em Populações Vulneráveis
Populações especiais como gestantes, idosos, crianças e pacientes imunossuprimidos requerem adaptações
no protocolo diagnóstico e terapêutico. Em gestantes, a teratogenicidade dos medicamentos deve ser
cuidadosamente avaliada, com preferência por agentes com maior segurança estabelecida no perfil
materno-fetal. Em idosos, a polifarmácia, as interações medicamentosas e a redução da reserva renal
e hepática exigem ajustes de dose e monitoramento mais frequente. Em pacientes pediátricos, doses
baseadas em peso corporal e considerações sobre o impacto no crescimento e desenvolvimento são
fundamentais para o planejamento terapêutico individualizado e centrado no paciente.
"""

TITULOS = [
    "Síndrome Inflamatória Multissistêmica — Diagnóstico e Manejo",
    "Doenças Autoimunes Sistêmicas — Classificação e Tratamento",
    "Insuficiência Cardíaca Congestiva — Fisiopatologia e Terapêutica",
    "Sepse e Choque Séptico — Protocolo de Ressuscitação",
    "Neoplasias Hematológicas — Diagnóstico Molecular e Imunoterapia",
]

# Repetir cada capítulo para atingir ~10.000-15.000 tokens
capitulos = [CAPITULO_TEMPLATE.format(n=i+1, titulo=TITULOS[i]) * 4 for i in range(5)]
contexto_rag = "\n\n".join(capitulos)

# Tokenizar
tokens = tokenizer(contexto_rag, return_tensors="pt")
n_tokens = tokens["input_ids"].shape[1]

print(f"=== MÉTRICA PASSO 2 ===")
print(f"Tokens no contexto RAG: {n_tokens:,}")
print(f"Caracteres no texto: {len(contexto_rag):,}")

## Passo 3: O Gargalo — Geração SEM KV Cache (baseline)

In [ ]:
# Preparar prompt clínico
system_prompt = "Você é um assistente médico especializado. Gere um resumo clínico conciso e preciso."
user_prompt = f"""Com base nos seguintes capítulos de manuais médicos recuperados:\n\n{contexto_rag[:8000]}\n\n\
Gere um resumo clínico de 500 palavras destacando os principais protocolos terapêuticos."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

print(f"Tokens de entrada (com template de chat): {input_ids.shape[1]:,}")

# ===== GERAÇÃO SEM CACHE (O PROBLEMA) =====
model.config.use_cache = False  # <-- FORÇA recálculo de Q, K, V a cada token

reset_cuda_stats()
torch.cuda.synchronize() if torch.cuda.is_available() else None

t_inicio = time.time()

with torch.no_grad():
    output_sem_cache = model.generate(
        input_ids,
        max_new_tokens=100,
        do_sample=False,
        use_cache=False,  # redundante mas explícito
    )

torch.cuda.synchronize() if torch.cuda.is_available() else None
t_sem_cache = time.time() - t_inicio

vram_pico_sem_cache = bytes_to_mb(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else 0

tokens_gerados = output_sem_cache.shape[1] - input_ids.shape[1]
texto_gerado = tokenizer.decode(output_sem_cache[0, input_ids.shape[1]:], skip_special_tokens=True)

print(f"\n=== MÉTRICA PASSO 3 — SEM KV CACHE ===")
print(f"Tokens gerados: {tokens_gerados}")
print(f"Tempo total de geração: {t_sem_cache:.2f}s")
print(f"Velocidade: {tokens_gerados / t_sem_cache:.2f} tokens/s")
print(f"Pico de VRAM: {vram_pico_sem_cache:.1f} MB")
print(f"\nTexto gerado (primeiros 300 chars):\n{texto_gerado[:300]}")

## Passo 4: A Engenharia de Otimização — KV Cache + FlashAttention-2

In [ ]:
# Liberar modelo anterior da VRAM
del model
reset_cuda_stats()

# Recarregar com FlashAttention-2
model_otimizado = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2",  # <-- OTIMIZAÇÃO DE HARDWARE (SRAM)
)

vram_modelo_otimizado = bytes_to_mb(torch.cuda.memory_allocated()) if torch.cuda.is_available() else 0
print(f"Modelo com FlashAttention-2 carregado. VRAM: {vram_modelo_otimizado:.1f} MB")

In [ ]:
# ===== GERAÇÃO COM KV CACHE + FLASHATTENTION-2 (A SOLUÇÃO) =====
model_otimizado.config.use_cache = True  # <-- ATIVA KV Cache

reset_cuda_stats()
torch.cuda.synchronize() if torch.cuda.is_available() else None

t_inicio_otimizado = time.time()

with torch.no_grad():
    output_otimizado = model_otimizado.generate(
        input_ids,
        max_new_tokens=100,
        do_sample=False,
        use_cache=True,  # <-- REUTILIZA K, V calculados
    )

torch.cuda.synchronize() if torch.cuda.is_available() else None
t_otimizado = time.time() - t_inicio_otimizado

vram_pico_otimizado = bytes_to_mb(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else 0

tokens_gerados_otimizado = output_otimizado.shape[1] - input_ids.shape[1]
texto_gerado_otimizado = tokenizer.decode(
    output_otimizado[0, input_ids.shape[1]:], skip_special_tokens=True
)

print(f"=== MÉTRICA PASSO 4 — COM KV CACHE + FLASHATTENTION-2 ===")
print(f"Tokens gerados: {tokens_gerados_otimizado}")
print(f"Tempo total de geração: {t_otimizado:.2f}s")
print(f"Velocidade: {tokens_gerados_otimizado / t_otimizado:.2f} tokens/s")
print(f"Pico de VRAM: {vram_pico_otimizado:.1f} MB")
print(f"\nTexto gerado (primeiros 300 chars):\n{texto_gerado_otimizado[:300]}")

## Relatório Comparativo Final

In [ ]:
import pandas as pd

speedup = t_sem_cache / t_otimizado if t_otimizado > 0 else float('inf')
reducao_vram = ((vram_pico_sem_cache - vram_pico_otimizado) / vram_pico_sem_cache * 100) if vram_pico_sem_cache > 0 else 0

df = pd.DataFrame({
    "Configuração": ["Sem KV Cache (baseline)", "KV Cache + FlashAttention-2"],
    "Tempo (s)": [f"{t_sem_cache:.2f}", f"{t_otimizado:.2f}"],
    "Velocidade (tok/s)": [
        f"{tokens_gerados / t_sem_cache:.2f}",
        f"{tokens_gerados_otimizado / t_otimizado:.2f}"
    ],
    "Pico VRAM (MB)": [f"{vram_pico_sem_cache:.1f}", f"{vram_pico_otimizado:.1f}"],
})

print(df.to_string(index=False))
print(f"\nSpeedup: {speedup:.2f}x mais rápido")
print(f"Redução de VRAM: {reducao_vram:.1f}%")
print(f"\nVRAM do modelo (4-bit QLoRA): {vram_modelo_usada:.1f} MB")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Lab 10 — Benchmark: Sem Cache vs KV Cache + FlashAttention-2", fontsize=13, fontweight="bold")

configs = ["Sem Cache\n(baseline)", "KV Cache +\nFlashAttn-2"]
cores = ["#e74c3c", "#2ecc71"]

# Gráfico 1: Tempo de geração
bars1 = axes[0].bar(configs, [t_sem_cache, t_otimizado], color=cores, edgecolor="black", width=0.5)
axes[0].set_title("Tempo de Geração (100 tokens)", fontsize=11)
axes[0].set_ylabel("Tempo (segundos)")
for bar, val in zip(bars1, [t_sem_cache, t_otimizado]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f"{val:.2f}s", ha="center", va="bottom", fontweight="bold")

# Gráfico 2: Pico de VRAM
bars2 = axes[1].bar(configs, [vram_pico_sem_cache, vram_pico_otimizado], color=cores, edgecolor="black", width=0.5)
axes[1].set_title("Pico de VRAM durante Geração", fontsize=11)
axes[1].set_ylabel("VRAM (MB)")
for bar, val in zip(bars2, [vram_pico_sem_cache, vram_pico_otimizado]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f"{val:.0f} MB", ha="center", va="bottom", fontweight="bold")

red_patch = mpatches.Patch(color="#e74c3c", label="Sem otimização")
green_patch = mpatches.Patch(color="#2ecc71", label="KV Cache + FlashAttn-2")
fig.legend(handles=[red_patch, green_patch], loc="lower center", ncol=2, fontsize=10)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig("benchmark_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico salvo em benchmark_results.png")